# Phishing Email Detection — Final Project

**Task:** Binary classification — Phishing (1) vs Legitimate (0)

**Primary Metric:** F1-Score

**Baseline to beat:** Logistic Regression + BoW → F1 = 0.8196


## 0. Setup & Dependencies


In [1]:
!pip install transformers datasets scikit-learn torch evaluate accelerate -q

import re, os, random, warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EvalPrediction
)
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
import scipy.sparse as sp

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import sys
print(f'Python version: {sys.version}')

# Central dict to track all metrics
ALL_METRICS = {}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 1. Dataset Loading & Exploration


In [ ]:
# Load the datasets
train_df = pd.read_csv('train.csv')
val_df   = pd.read_csv('val.csv')
test_df  = pd.read_csv('test.csv')

# Explore datasets
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Columns: {train_df.columns.tolist()}')
print(f'\nLabel distribution (train):\n{train_df["label"].value_counts()}')
train_df.head(3)


Train: 10500 | Val: 2250 | Test: 2250
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls', 'annotation1', 'annotation2']

Label distribution (train):
label
1    5250
0    5250
Name: count, dtype: int64


,sender,receiver,date,subject,body,label,urls,annotation1,annotation2
0,Leanna Hooks <kglgilqgyduehc@avnetgroup.com>,jreitme@enron.com,"Fri, 04 Jan 2002 21:33:00 -0200",A new major market score each week,"""Stock Watch A|ert"" this morning are Wysak Pet...",1,0,NaN,NaN
1,al@mpsc.ph,<ilug@linux.ie>,"Wed, 21 Aug 2002 14:52:11 +0800 (PHT)",[ILUG] dial-on-demand,Could you please help me how to set up a dial-...,0,1,0.0,NaN
2,wantEnjoy <hand@verticalcircuits.com>,theorize@plg.uwaterloo.ca,"Wed, 02 May 2007 10:04:54 +0900",Windows Vista Home Links,"Logotry cool, see companies, building products...",1,0,NaN,NaN


## 2. Preprocessing & Feature Engineering


In [ ]:
URGENT_PATTERN = (
    r'\b(urgent|immediately|verify|account|suspended|click|confirm|update|'
    r'free|winner|congratulations|password|login|bank|credit|debit|expire|'
    r'limited|act now|risk|alert|security|validate|authorize)\b'
)

# Clean text by removing HTML, URLs, and normalizing whitespace/newlines
def clean_text(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'http\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Extract sender domain from email address
def get_sender_domain(addr):
    if pd.isna(addr): return 'unknown'
    m = re.search(r'@([\w\.-]+)', str(addr))
    return m.group(1).lower() if m else 'unknown'

# Preprocess the DataFrame by cleaning text, extracting features, and combining into a single 'text' column
def preprocess(df):
    d = df.copy()
    for col in ['subject', 'body']:
        d[col] = d[col].apply(clean_text) if col in d.columns else ''

    sender_domain = d['sender'].apply(get_sender_domain) if 'sender' in d.columns else 'unknown'

    # Give more weight to subject by repeating it, and include sender domain as a strong signal
    d['text'] = (
        'domain: ' + sender_domain + ' ' +
        d['subject'].fillna('') + ' ' +
        d['subject'].fillna('') + ' ' +
        d['body'].fillna('')
    )

    # Extract additional features
    d['body_len']      = d['body'].str.len().fillna(0)
    d['subject_len']   = d['subject'].str.len().fillna(0)
    d['url_count']     = d['body'].str.count('URL').fillna(0)
    d['exclaim_count'] = d['body'].str.count('!').fillna(0)
    d['dollar_count']  = d['body'].str.count(r'\$').fillna(0)
    d['urgent_words']  = d['body'].str.lower().str.count(URGENT_PATTERN).fillna(0)
    d['has_url_flag']  = d['urls'].fillna(0) if 'urls' in d.columns else 0
    return d

# Preprocess all datasets
train_df = preprocess(train_df)
val_df   = preprocess(val_df)
test_df  = preprocess(test_df)

print('Preprocessing complete')
print(f'Sample combined text:\n{train_df["text"].iloc[0][:300]}...')

Preprocessing complete
Sample combined text:
domain: avnetgroup.com A new major market score each week A new major market score each week "Stock Watch A|ert" this morning are Wysak Petroleum (WYSK), Key Energy Services, Inc. (Pink Sheets: KEGS), Medify So|utions (MFYS), Sequoia Interests Corporation (SQNC). Wysak Petro|eum (WYSK) Current Price...


## 3. TF-IDF + Logistic Regression Baseline


In [ ]:
# Custom transformer to extract numeric features for the pipeline
class NumericFeatures(BaseEstimator, TransformerMixin):
    COLS = ['body_len','subject_len','url_count','exclaim_count','dollar_count','urgent_words','has_url_flag']
    def fit(self, X, y=None): return self
    def transform(self, X): return sp.csr_matrix(X[self.COLS].values.astype(float))

# Custom transformer to select the text column for vectorization
class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, col): self.col = col
    def fit(self, X, y=None): return self
    def transform(self, X): return X[self.col]

# Word n-grams (1-3) + character n-grams (3-5) + hand-crafted numeric features
lr_pipeline = Pipeline([
    ('features', FeatureUnion([
        ('word_tfidf', Pipeline([
            ('sel',   TextSelector('text')),
            ('tfidf', TfidfVectorizer(sublinear_tf=True, max_features=80000, ngram_range=(1,3), min_df=2, strip_accents='unicode')),
        ])),
        ('char_tfidf', Pipeline([
            ('sel',   TextSelector('text')),
            ('tfidf', TfidfVectorizer(sublinear_tf=True, max_features=30000, ngram_range=(3,5), analyzer='char_wb', min_df=3)),
        ])),
        ('numeric', NumericFeatures()),
    ])),
    ('clf', LogisticRegression(C=5.0, max_iter=1000, solver='lbfgs', random_state=SEED)),
])

# Train the Logistic Regression pipeline and evaluate on validation set
lr_pipeline.fit(train_df, train_df['label'])
val_preds_lr = lr_pipeline.predict(val_df)

print('--- TF-IDF + LR Validation Metrics ---')
print(f"F1:        {f1_score(val_df['label'], val_preds_lr):.4f}")
print(f"Accuracy:  {accuracy_score(val_df['label'], val_preds_lr):.4f}")
print(f"Precision: {precision_score(val_df['label'], val_preds_lr):.4f}")
print(f"Recall:    {recall_score(val_df['label'], val_preds_lr):.4f}")

ALL_METRICS['TF-IDF + LR'] = {
    'validation': {
        'f1':        f1_score(val_df['label'], val_preds_lr),
        'accuracy':  accuracy_score(val_df['label'], val_preds_lr),
        'precision': precision_score(val_df['label'], val_preds_lr),
        'recall':    recall_score(val_df['label'], val_preds_lr),
    }
}

--- TF-IDF + LR Validation Metrics ---
F1:        0.9538
Accuracy:  0.9538
Precision: 0.9530
Recall:    0.9547


## 4. Fine-tuned Transformer (RoBERTa)


In [ ]:
# Build a HuggingFace DatasetDict from the preprocessed DataFrames
# Test set has no labels so we create a dummy label column of zeros
test_df_hf = test_df.copy()
test_df_hf['label'] = 0

dataset = DatasetDict({
    'train':      Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['text','label']].reset_index(drop=True)),
    'test':       Dataset.from_pandas(test_df_hf[['text','label']].reset_index(drop=True)),
})

id2label = {0: 'Legitimate', 1: 'Phishing'}
label2id = {'Legitimate': 0, 'Phishing': 1}

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10500
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2250
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2250
    })
})


In [ ]:
# Load the tokenizer and model for roBERTa-base
MODEL_NAME = 'roberta-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    problem_type='single_label_classification',
)
print(f'Loaded {MODEL_NAME} — parameters: {model.num_parameters():,}')

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded roberta-base — parameters: 124,647,170


In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=512,
    )

# Tokenize the datasets and set format for PyTorch
tokenized_ds = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized_ds.set_format('torch')

print(f'Columns: {tokenized_ds["train"].column_names}')

Map:   0%|          | 0/10500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Columns: ['label', 'input_ids', 'attention_mask']


In [ ]:
# Function for evaluating metrics during training and validation
def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    pred_labels = np.argmax(preds, axis=1)
    true_labels = p.label_ids
    return {
        'f1':        f1_score(true_labels, pred_labels),
        'accuracy':  accuracy_score(true_labels, pred_labels),
        'precision': precision_score(true_labels, pred_labels),
        'recall':    recall_score(true_labels, pred_labels),
    }

# Set up the Trainer with training arguments and start training
training_args = TrainingArguments(
    output_dir='./roberta_phishing',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.15,
    weight_decay=0.01,
    learning_rate=1e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    compute_metrics=compute_metrics,
)

trainer.train()
eval_results = trainer.evaluate()
print(eval_results)

ALL_METRICS[MODEL_NAME] = {
    'validation': {
        'f1':        eval_results.get('eval_f1'),
        'accuracy':  eval_results.get('eval_accuracy'),
        'precision': eval_results.get('eval_precision'),
        'recall':    eval_results.get('eval_recall'),
    }
}

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1,Accuracy,Precision,Recall
1,0.319055,0.279752,0.927619,0.932444,0.998974,0.865778
2,0.096823,0.083397,0.985676,0.985778,0.992786,0.978667
3,0.051376,0.080376,0.984862,0.984889,0.986619,0.983111
4,0.022056,0.103284,0.982014,0.982222,0.993631,0.970667
5,0.011867,0.094643,0.984753,0.984889,0.993665,0.976000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': 0.0835333913564682, 'eval_f1': 0.9856759176365264, 'eval_accuracy': 0.9857777777777778, 'eval_precision': 0.9927862939585211, 'eval_recall': 0.9786666666666667, 'eval_runtime': 70.992, 'eval_samples_per_second': 31.694, 'eval_steps_per_second': 1.0, 'epoch': 5.0}


In [ ]:
# Validation metrics
print('--- RoBERTa Validation Metrics ---')
print(f"F1:        {eval_results.get('eval_f1'):.4f}")
print(f"Accuracy:  {eval_results.get('eval_accuracy'):.4f}")
print(f"Precision: {eval_results.get('eval_precision'):.4f}")
print(f"Recall:    {eval_results.get('eval_recall'):.4f}")

# Get raw logits on validation and test for ensemble
val_output  = trainer.predict(tokenized_ds['validation'])
test_output = trainer.predict(tokenized_ds['test'])

# Convert logits to probabilities using softmax and take the phishing class probability
import torch.nn.functional as F
bert_val_probs  = F.softmax(torch.Tensor(val_output.predictions),  dim=1)[:, 1].numpy()
bert_test_probs = F.softmax(torch.Tensor(test_output.predictions), dim=1)[:, 1].numpy()

print(f'\nVal probs sample: {bert_val_probs[:5]}')

--- RoBERTa Validation Metrics ---
F1:        0.9857
Accuracy:  0.9858
Precision: 0.9928
Recall:    0.9787



Val probs sample: [2.4086921e-04 9.9947816e-01 9.9964857e-01 2.0158535e-04 9.9964595e-01]


## 5. Ensemble (TF-IDF LR + RoBERTa)


In [ ]:
# Get LR probabilities
lr_val_probs  = lr_pipeline.predict_proba(val_df)[:, 1]
lr_test_probs = lr_pipeline.predict_proba(test_df)[:, 1]

# Sweep the RoBERTa blend weight (alpha) on validation to find the best F1
best_alpha, best_ens_f1 = 0.5, 0.0
for alpha in np.arange(0.0, 1.05, 0.05):
    ens_preds = (alpha * bert_val_probs + (1 - alpha) * lr_val_probs >= 0.5).astype(int)
    ens_f1 = f1_score(val_df['label'], ens_preds)
    if ens_f1 > best_ens_f1:
        best_ens_f1 = ens_f1
        best_alpha  = alpha

print(f'Best alpha (RoBERTa weight): {best_alpha:.2f} | Ensemble Val F1: {best_ens_f1:.4f}')

# Create final ensemble predictions on validation set using the best alpha
final_val_probs = best_alpha * bert_val_probs + (1 - best_alpha) * lr_val_probs
final_val_preds = (final_val_probs >= 0.5).astype(int)

print('\n--- Ensemble Validation Metrics ---')
print(f"F1:        {f1_score(val_df['label'], final_val_preds):.4f}")
print(f"Accuracy:  {accuracy_score(val_df['label'], final_val_preds):.4f}")
print(f"Precision: {precision_score(val_df['label'], final_val_preds):.4f}")
print(f"Recall:    {recall_score(val_df['label'], final_val_preds):.4f}")

ALL_METRICS['Ensemble'] = {
    'validation': {
        'f1':        f1_score(val_df['label'], final_val_preds),
        'accuracy':  accuracy_score(val_df['label'], final_val_preds),
        'precision': precision_score(val_df['label'], final_val_preds),
        'recall':    recall_score(val_df['label'], final_val_preds),
    }
}


Best alpha (RoBERTa weight): 0.55 | Ensemble Val F1: 0.9861

--- Ensemble Validation Metrics ---
F1:        0.9861
Accuracy:  0.9862
Precision: 0.9928
Recall:    0.9796


## 6. Threshold Tuning


In [ ]:
# Tune the decision threshold on validation
best_threshold, best_thresh_f1 = 0.5, 0.0
for thresh in np.arange(0.30, 0.71, 0.01):
    preds = (final_val_probs >= thresh).astype(int)
    tf1 = f1_score(val_df['label'], preds)
    if tf1 > best_thresh_f1:
        best_thresh_f1 = tf1
        best_threshold = thresh

print(f'Best threshold: {best_threshold:.2f} | Val F1: {best_thresh_f1:.4f}')

# Create final ensemble predictions on validation set using the best threshold
tuned_val_preds = (final_val_probs >= best_threshold).astype(int)
print('\n--- Threshold-tuned Ensemble Validation Metrics ---')
print(f"F1:        {f1_score(val_df['label'], tuned_val_preds):.4f}")
print(f"Accuracy:  {accuracy_score(val_df['label'], tuned_val_preds):.4f}")
print(f"Precision: {precision_score(val_df['label'], tuned_val_preds):.4f}")
print(f"Recall:    {recall_score(val_df['label'], tuned_val_preds):.4f}")

Best threshold: 0.46 | Val F1: 0.9861

--- Threshold-tuned Ensemble Validation Metrics ---
F1:        0.9861
Accuracy:  0.9862
Precision: 0.9928
Recall:    0.9796


## 7. Generate Submission


In [ ]:
# Apply the best alpha and threshold to test set
final_test_probs = best_alpha * bert_test_probs + (1 - best_alpha) * lr_test_probs
final_test_preds = (final_test_probs >= best_threshold).astype(int)

# Sanity check on test predictions
assert len(final_test_preds) == 2250, f'Expected 2250 rows, got {len(final_test_preds)}'

# Create CSV submission file
submission = pd.DataFrame({'label': final_test_preds})
submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv — {len(submission)} rows')
print(submission['label'].value_counts())

Saved submission.csv — 2250 rows
label
0    1134
1    1116
Name: count, dtype: int64


## 8. Results Summary

| Model | Val F1 | Val Accuracy | Val Precision | Val Recall |
|---|---|---|---|---|
| Baseline (Majority) | 0.6667 | 0.5000 | 0.5000 | 1.0000 |
| LR + BoW (Baseline) | 0.8196 | 0.8116 | 0.7861 | 0.8560 |
| Ensemble (Mine) | 0.9844 | 0.9844 | 0.9884 | 0.9804 |